# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:8] + "..." if hf_token else "NOT FOUND")

Token loaded: hf_DCVCv...


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Tables defined.")

Tables defined.


In [3]:
feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_ctr: (153559, 4)
feb_position: (151956, 2)


In [4]:
feb_position["feb_avg_position"].describe()

,feb_avg_position
count,151956.000000
mean,14.133842
std,15.256815
min,0.057700
25%,5.348455
50%,8.647976
75%,16.500000
max,633.000000


feb_ctr table (153,559 rows, 4 columns): content_hash_id (the page identifier), feb_clicks, feb_impressions, feb_ctr (the computed ratio).
feb_position table (151,956 rows, 2 columns): content_hash_id, feb_avg_position.

skew: one extreme (rare) value drags the average far from where most of the data actually sits. Same thing here — most pages rank reasonably (median ~8.65), but a few pages ranking terribly (position 633!) drag the average up to 14.1, making it look worse than what's typical.

In [5]:
# Merge position and CTR data together first, so we can compare them per page
signal_check_1 = feb_position.merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="inner")

# Build position buckets
def position_bucket(pos):
    if pos <= 10:
        return "top"
    elif pos <= 30:
        return "middle"
    else:
        return "poor"

signal_check_1["position_bucket"] = signal_check_1["feb_avg_position"].apply(position_bucket)

# Bucket table: average CTR per position bucket, with n (count) printed
bucket_table = signal_check_1.groupby("position_bucket").agg(
    mean_ctr=("feb_ctr", "mean"),
    n=("feb_ctr", "count")
).reindex(["top", "middle", "poor"])  # keep logical order

print(bucket_table)

                 mean_ctr      n
position_bucket                 
top              0.006083  88152
middle           0.002983  46293
poor             0.002361  17511


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal 1: CTR vs. Position (linked to FlyRank's real CTR-fix flag)

**Hypothesis:** Pages ranked well (lower average position) should get meaningfully higher
click-through rate (CTR) than poorly-ranked pages — this is the assumption behind FlyRank's
real CTR-fix flag, and matches what we already found in the starter CSV (notebook 1,
Discovery B).

**Bucket table (February 2026, warehouse data):**

| position_bucket | mean_ctr | n      |
|---|---|---|
| top (≤10)        | 0.006083 | 88,151 |
| middle (11-30)   | 0.002983 | 46,294 |
| poor (30+)       | 0.002361 | 17,511 |

**Verdict: CONFIRMED.** CTR drops as position worsens, exactly as expected — top-ranked
pages get roughly 2x the CTR of middle-ranked pages, and middle pages get somewhat higher
CTR than poorly-ranked pages. This holds across a large sample (n=151,956 total pages with
real GSC position and CTR data), giving us confidence this signal is real and safe to build
our rule on.

(Note: these raw CTR values, e.g. 0.006, appear on a different scale than the starter CSV's
CTR column, e.g. 0.15-0.35 — likely due to different scaling conventions between the two
datasets. The relative pattern across buckets, which is what matters for this verdict,
holds regardless of scale.)

In [9]:
# Rebuild March impressions (needed to recreate is_declining, same as w03)
march_impressions = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Rebuild the label: Feb vs March comparison (same 20%-drop threshold as w03)
trend_data = feb_ctr[["content_hash_id"]].merge(
    con.sql(f"""
        SELECT content_hash_id, SUM(gsc_impressions) AS feb_impressions
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df(),
    on="content_hash_id", how="inner"
).merge(march_impressions, on="content_hash_id", how="inner")

trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print(trend_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(134238, 4)


In [13]:
# --- 3. Rebuild February features (impressions, CTR, position) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_impressions: (153559, 2)
feb_ctr: (153559, 4)
feb_position: (151956, 2)


In [14]:
# --- 5. Check feb_impressions distribution (volume signal, next step) ---
trend_data["feb_impressions"].describe()

,feb_impressions
count,134238.000000
mean,1324.283601
std,4275.575393
min,1.000000
25%,22.000000
50%,168.000000
75%,962.000000
max,203401.000000


In [16]:
# Build volume buckets using real percentiles from trend_data
def volume_bucket(imp):
    if imp <= 168:       # <= median
        return "low"
    elif imp <= 962:      # <= 75th percentile
        return "medium"
    else:
        return "high"

trend_data["volume_bucket"] = trend_data["feb_impressions"].apply(volume_bucket)

bucket_table_3 = trend_data.groupby("volume_bucket").agg(
    decline_rate=("is_declining", "mean"),
    n=("is_declining", "count")
).reindex(["low", "medium", "high"])

print(bucket_table_3)

               decline_rate      n
volume_bucket                     
low                0.224205  67135
medium             0.193890  33550
high               0.157005  33553


## Signal 2: Volume vs. Decline (linked to FlyRank's real quick-win logic)

**Hypothesis (original):** Pages with high February impressions (volume) would be MORE likely
to be declining, due to greater scrutiny/competition at scale.

**Bucket table (February 2026 impressions vs. Feb→March decline label):**

| volume_bucket | decline_rate | n      |
|---|---|---|
| low            | 0.224205     | 67,135 |
| medium         | 0.193890     | 33,550 |
| high           | 0.157005     | 33,553 |

**Verdict: OPPOSITE.** High-volume pages are actually LESS likely to be declining (15.7%)
than low-volume pages (22.4%) — a clean, monotonic pattern across a large sample
(n=134,238 total). This contradicts our original hypothesis, but is a real, usable finding:
popular, high-traffic pages appear more stable, likely because established pages have
proven staying power (similar to the pattern found by the decision tree in notebook 02).

**How this reshapes our rule:** Since high volume does not predict decline, we will NOT use
volume as a way to *find* declining pages. Instead, we use it to *prioritize among already-
declining pages*: if a page is declining, higher volume means fixing it has a bigger payoff
(more traffic at stake), making it a better "quick win" once flagged — not a standalone
signal that a page needs review in the first place.

## My Rule

**Plain words:** Flag pages that rank well (good position) but have surprisingly low CTR —
this is the CONFIRMED signal from Signal 1, indicating the listing itself (title/meta) may be
underperforming despite good visibility. Among flagged pages, prioritize those with higher
February volume — per Signal 2's OPPOSITE finding, high volume doesn't predict decline, but
it does mean a fix pays off more, since more traffic is already at stake.

**Reason code:** `ctr_underperformance_high_value` — assigned when a page has a good position
(≤10) but below-expected CTR for that tier, weighted by its traffic volume.

**Action label:** `review_title_meta` — the recommended human action: review and likely rewrite
the page's title/meta description, since the page already has visibility (good position) but
isn't converting it into clicks.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.